# Same-source collection groups

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/outputs/Document_Outputs_Group.ipynb)

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

In [ ]:
import tempfile
from klygo import files, outputs, metrics
tutorial_workspace = tempfile.TemporaryDirectory()
root = files.path(tutorial_workspace.name)
pred = outputs.Detections([outputs.Detection(
    [outputs.Box((0, 0, 20, 20), label="cat", score=0.9)],
    source_id="image-a", image_size=(100, 100),
)])
target = outputs.Detections([outputs.Detection(
    [outputs.Box((0, 0, 20, 20), label="cat")],
    source_id="image-a", image_size=(100, 100),
)])


## Jump to an API

[outputs.group](#outputsgroup) · [outputs.DetectionGroup](#outputsDetectionGroup)

### outputs.group

`outputs.group(*collections) -> DetectionGroup`

- Parameters: distinct materialized Detections collections with equal frame identities and image sizes.
- Returns: reference group supporting cuda, cpu, astype, keep, drop, relabel, compact, pack and unpack.
- Frame order and annotations can differ. No data is merged or copied by grouping.
- TypeError for non-Detections inputs; ValueError for no members, duplicates, streams, or source mismatches.

**Basic use**


In [ ]:
collections = outputs.group(pred, target)
collections.astype(32).pack().unpack()
print(pred.dtype, target.dtype, collections[0] is pred)
print(metrics.precision_recall(pred, target))


**Practical variation**: one device operation for all members. GPU metrics follows their device.

In [ ]:
try:
    import cupy as cp
    cuda_available = cp.cuda.runtime.getDeviceCount() > 0
except (ImportError, RuntimeError):
    cuda_available = False
if cuda_available:
    collections.cuda(0).astype(32)
    evaluator = metrics.compose(metrics.precision_recall(), metrics.map_score(), metrics.calibration())
    report = metrics.evaluate(pred, target, evaluator)
    print(pred.device, target.device, report)
    files.save(root / "gpu-report.json", report, overwrite=True, verbose=False)
    collections.cpu()
else:
    print("CUDA unavailable; CPU group operations remain supported")


**Expected error and recovery**: different sources cannot form a group.

In [ ]:
other = outputs.Detections([outputs.Detection([], source_id="other", image_size=(100, 100))])
try:
    outputs.group(pred, other)
except ValueError as error:
    print(error)
collections = outputs.group(pred, target)


### outputs.DetectionGroup

`outputs.DetectionGroup(*collections)` constructs the same reference group.
Members, iteration, indexing and len expose the original collections. Every
operation returns the group for chaining and revalidates source alignment.
Individual collection methods remain supported. Operations are sequential;
a failure can leave earlier members already changed. Compact uses each member's
own class map; matching compares label strings.

**Basic use**


In [ ]:
collections = outputs.DetectionGroup(pred, target)
collections.relabel({"cat": "animal"}).compact()
print(pred[0].labels, target[0].labels)


**Practical variation**: iterable labels are shared across all members.

In [ ]:
collections.keep(name for name in ["animal"])
collections.drop(["missing"])
pred.astype(64)
print(pred.dtype, target.dtype)


**Expected error and recovery**: duplicate references are rejected.

In [ ]:
try:
    outputs.DetectionGroup(pred, pred)
except ValueError as error:
    print(error)
collections = outputs.DetectionGroup(pred, target)


## Put it together

In [ ]:
collections.astype(32)
evaluator = metrics.compose(metrics.precision_recall(), metrics.per_class(), metrics.per_image())
report = metrics.evaluate(pred, target, evaluator)
files.save(root / "report.json", report, overwrite=True, verbose=False)
print(report)


## Named members

Two positional collections use `pred`, `truth`. Other counts use `set_1`,
`set_2`, and so on. Names follow input order. A mapping assigns explicit names.
Iteration yields collections; name and integer indexing both select a member.

`rename(mapping)` changes member names simultaneously and returns this group.
Unmentioned names are preserved and swaps are supported. Unknown source names,
empty names, or final collisions fail before names change. Member order,
references, annotations, and devices remain unchanged.


In [ ]:
automatic = outputs.group(pred, target)
assert automatic["pred"] is pred and automatic["truth"] is target
assert list(automatic.keys()) == ["pred", "truth"]
named = outputs.group({"model": pred, "ground_truth": target})
assert list(named.keys()) == ["model", "ground_truth"]
assert tuple(named.values()) == (pred, target)
print(metrics.evaluate(named["model"], named["ground_truth"]))
named.rename({"model": "pred", "ground_truth": "truth"})
assert named["pred"] is pred and named["truth"] is target
named.rename({"pred": "truth", "truth": "pred"})
assert named["truth"] is pred
try:
    named.rename({"pred": "truth"})
except ValueError as error:
    print(error)
assert list(named.keys()) == ["truth", "pred"]


## Remember

Group operations mutate the original members; contents remain independent.
Sources must align by identity and size before every operation, not by position.
GPU support requires CuPy and CUDA. Metrics inputs must share a device, and
reports remain ordinary Python data. Small workloads may be faster on CPU.

[Overview](Document_Outputs_Overview.ipynb) · [Detection arrays](Document_Outputs_Detect.ipynb) · [Metrics](../metrics/Document_Metrics_Overview.ipynb) · [Files](../files/Document_Files_Overview.ipynb)


In [ ]:
tutorial_workspace.cleanup()